# entity_action_bc_v1 — Sequence Auxiliary Head Training

**Model:** `entity_action_bc_v1_20260406_seq_run1`  
**Script:** `train_entity_action.py` (entity-centric BC policy)

## What this run adds
- `--predict-turn-sequence` — LSTM sequence head that learns to predict the ordered
  turn-event token sequence (moves, damage, switches, etc.) conditioned on
  state + both player actions. Sequence loss starts ~log(vocab_size) and drops.
- `--sequence-weight 0.1` — flat auxiliary weight (same as model5 policy run)
- `--sequence-hidden-dim 128` — LSTM hidden size for sequence head
- `--max-seq-len 32` — max token sequence length per turn

Other auxiliary heads also enabled: `--predict-turn-outcome`, `--predict-value`.

## Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells top-to-bottom (or `Runtime → Run all`)
3. Cell 8 downloads artifacts to your local browser at the end

In [ ]:
# ── 1. Verify GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected — go to Runtime → Change runtime type → T4 GPU')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF sees:', gpus if gpus else 'NO GPU — check runtime type')

In [ ]:
# ── 2. Clone repo (feature branch with entity + sequence head work) ─────────
import os

REPO_URL = 'https://github.com/AlterProgramming/Pokemon-Showdown-Sim.git'
BRANCH   = 'feature/turn-event-tokenizer-and-sequence-head'
REPO_DIR = '/content/Pokemon-Showdown-Sim'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!git log --oneline -3

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────
# Colab ships with TF pre-installed; only add kagglehub and pin keras.
import tensorflow as tf
print('Colab TF version:', tf.__version__)

!pip install -q kagglehub

# Colab TF 2.15/2.16 ships with keras 2.x; train_entity_action.py requires keras 3.x.
import importlib
try:
    import keras
    major = int(keras.__version__.split('.')[0])
    if major < 3:
        print(f'keras {keras.__version__} is too old — upgrading to 3.x')
        !pip install -q 'keras>=3.0'
        importlib.invalidate_caches()
    else:
        print(f'keras {keras.__version__} OK')
except ImportError:
    !pip install -q 'keras>=3.0'

print()
print('If this is your first run, use Runtime → Restart session, then re-run from cell 1.')

In [ ]:
# ── 4. Download battle data (public — no credentials needed) ────────────────
import kagglehub, os, glob

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print(f'Downloading {DATASET} ...')
data_path = kagglehub.dataset_download(DATASET)

json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')

In [ ]:
# ── 5. Training configuration ──────────────────────────────────────────────
import os
os.chdir('/content/Pokemon-Showdown-Sim')

# --- Identity ---
MODEL_NAME = 'entity_action_bc_v1_20260406_seq_run1'
OUTPUT_DIR = '/content/output'

# --- Data ---
MAX_BATTLES = 5000          # number of battle logs to load from the dataset

# --- Optimisation ---
EPOCHS        = 30
BATCH_SIZE    = 256         # entity model uses 256 (smaller than policy 512)
LEARNING_RATE = 0.001
PATIENCE      = 3           # early-stopping patience on val top-3

# --- Architecture ---
HIDDEN_DIM   = 256
DEPTH        = 3
DROPOUT      = 0.1

# --- Auxiliary head weights ---
PREDICT_TURN_OUTCOME = True   # transition auxiliary head
PREDICT_VALUE        = True   # win-probability auxiliary head
PREDICT_TURN_SEQUENCE = True  # NEW: sequence auxiliary head

TRANSITION_WEIGHT    = 0.25
VALUE_WEIGHT         = 0.25
SEQUENCE_WEIGHT      = 0.1   # flat-run value per spec

# --- Sequence head architecture ---
SEQUENCE_HIDDEN_DIM = 128    # LSTM hidden size
MAX_SEQ_LEN         = 32    # max event tokens per turn

# --- Return weighting ---
POLICY_RETURN_WEIGHTING    = 'exp'
POLICY_RETURN_WEIGHT_SCALE = 0.75
POLICY_RETURN_WEIGHT_MIN   = 0.25
POLICY_RETURN_WEIGHT_MAX   = 4.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Model name :', MODEL_NAME)
print('Output dir :', OUTPUT_DIR)
print('Max battles:', MAX_BATTLES)
print('Sequence head enabled:', PREDICT_TURN_SEQUENCE)
print('  sequence_weight    :', SEQUENCE_WEIGHT)
print('  sequence_hidden_dim:', SEQUENCE_HIDDEN_DIM)
print('  max_seq_len        :', MAX_SEQ_LEN)

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Pokemon-Showdown-Sim'

# Ensure repo root is on PYTHONPATH so bare imports inside the subprocess
# (e.g. `from TurnEventTokenizer import ...`) resolve correctly.
env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_entity_action.py',
    data_path,
    '--model-name',            MODEL_NAME,
    '--output-dir',            OUTPUT_DIR,
    '--max-battles',           str(MAX_BATTLES),
    '--epochs',                str(EPOCHS),
    '--batch-size',            str(BATCH_SIZE),
    '--learning-rate',         str(LEARNING_RATE),
    '--patience',              str(PATIENCE),
    '--hidden-dim',            str(HIDDEN_DIM),
    '--depth',                 str(DEPTH),
    '--dropout',               str(DROPOUT),
    # auxiliary heads
    '--predict-turn-outcome',
    '--predict-value',
    '--predict-turn-sequence',
    '--transition-weight',     str(TRANSITION_WEIGHT),
    '--value-weight',          str(VALUE_WEIGHT),
    '--sequence-weight',       str(SEQUENCE_WEIGHT),
    '--sequence-hidden-dim',   str(SEQUENCE_HIDDEN_DIM),
    '--max-seq-len',           str(MAX_SEQ_LEN),
    # return weighting
    '--policy-return-weighting',    POLICY_RETURN_WEIGHTING,
    '--policy-return-weight-scale', str(POLICY_RETURN_WEIGHT_SCALE),
    '--policy-return-weight-min',   str(POLICY_RETURN_WEIGHT_MIN),
    '--policy-return-weight-max',   str(POLICY_RETURN_WEIGHT_MAX),
    '--seed', '42',
]

print('Command:', ' '.join(cmd[:4]), '...')
print('=' * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
    env=env,
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('=' * 70)
print('Exit code:', proc.returncode)

In [ ]:
# ── 7. Inspect artifacts ───────────────────────────────────────────────────
import os, json

artifacts = sorted(os.listdir(OUTPUT_DIR))
print(f'Saved artifacts in {OUTPUT_DIR}:')
for f in artifacts:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:60s}  {size / 1024:.1f} KB')

# Print key metadata fields from training_metadata JSON
meta_path = os.path.join(OUTPUT_DIR, f'training_metadata_{MODEL_NAME}.json')
if not os.path.exists(meta_path):
    candidates = [f for f in artifacts if f.startswith('training_metadata')]
    meta_path = os.path.join(OUTPUT_DIR, candidates[0]) if candidates else None

if meta_path and os.path.exists(meta_path):
    meta = json.loads(open(meta_path).read())
    print()
    print('Key metadata:')
    keys_of_interest = (
        'model_name', 'epochs_completed', 'train_examples', 'val_examples',
        'policy_param_count', 'training_param_count',
        'sequence_vocab_size', 'max_seq_len',
        'sequence_weight', 'transition_weight', 'value_weight',
        'training_regime',
    )
    for k in keys_of_interest:
        if k in meta:
            print(f'  {k}: {meta[k]}')
else:
    print('No training_metadata JSON found — check exit code above.')

In [ ]:
# ── 8. Download artifacts to local machine ─────────────────────────────────
# Each file is downloaded to your browser individually.
# The .keras file is the deployable model; the JSON files are required at
# inference time (vocabs) and for experiment tracking (metadata).
from google.colab import files
import os

DOWNLOAD = [
    f'model_{MODEL_NAME}.keras',
    f'action_vocab_{MODEL_NAME}.json',
    f'action_context_vocab_{MODEL_NAME}.json',
    f'sequence_vocab_{MODEL_NAME}.json',
    f'training_metadata_{MODEL_NAME}.json',
]

for fname in DOWNLOAD:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        print(f'Downloading {fname} ...')
        files.download(fpath)
    else:
        print(f'Not found (may use a different name): {fname}')

print()
print('All files available in output dir:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}  ({size / 1024:.1f} KB)')

In [ ]:
# ── 13. Push artifacts to GCS ─────────────────────────────────────────────
# Uploads trained artifacts to gs://artifacts-model-serving/artifacts/{family}/{run_id}/
# Uses your logged-in Colab Google account — no secrets required.
import os, glob

BUCKET     = 'artifacts-model-serving'
GCS_PREFIX = f'artifacts/entity_action_bc_v1/{RUN_ID}'

try:
    from google.colab import auth
    from google.cloud import storage

    auth.authenticate_user()  # one-time browser prompt per Colab session
    client = storage.Client(project='pokemon-battle-agent-494015')
    bucket = client.bucket(BUCKET)

    pushed = []
    for local_path in glob.glob(os.path.join(OUTPUT_DIR, '*')):
        if not os.path.isfile(local_path):
            continue
        fname     = os.path.basename(local_path)
        blob_name = f'{GCS_PREFIX}/{fname}'
        bucket.blob(blob_name).upload_from_filename(local_path)
        pushed.append(blob_name)
        print(f'  ✓ {blob_name}')

    print(f'\nPushed {len(pushed)} files → gs://{BUCKET}/{GCS_PREFIX}/')
    print('Run /gcs-push locally to sync model_registry.json.')

except Exception as e:
    print(f'GCS push failed: {e}')
